### 🧠 What is Query Decomposition?
Query decomposition is the process of taking a complex, multi-part question and breaking it into simpler, atomic sub-questions that can each be retrieved and answered individually.

#### ✅ Why Use Query Decomposition?

- Complex queries often involve multiple concepts

- LLMs or retrievers may miss parts of the original question

- It enables multi-hop reasoning (answering in steps)

- Allows parallelism (especially in multi-agent frameworks)

In [4]:
from langchain_classic.chat_models import init_chat_model
from langchain_classic.prompts import PromptTemplate
from langchain_classic.document_loaders import TextLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.runnables import RunnableSequence

In [5]:
# Step 1: Load and embed the document
loader = TextLoader("langchain_crewai_dataset.txt")
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(docs)

embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embedding)
retriever = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 4, "lambda_mult": 0.7})

In [6]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

llm=init_chat_model(model="ollama:llama3.2:latest")
llm

ChatOllama(model='llama3.2:latest')

In [7]:
# Step 3: Query decomposition
decomposition_prompt = PromptTemplate.from_template("""
You are an AI assistant. Decompose the following complex question into 2 to 4 smaller sub-questions for better document retrieval.

Question: "{question}"

Sub-questions:
""")
decomposition_chain = decomposition_prompt | llm | StrOutputParser()

In [8]:
query = "How does LangChain use memory and agents compared to CrewAI?"
decomposition_question=decomposition_chain.invoke({"question": query})


In [9]:
print(decomposition_question)

Here are 4 smaller sub-questions that can help with retrieving relevant information about LangChain, CrewAI, memory usage, and agent comparison:

1. **What is the primary difference in how LangChain manages memory compared to CrewAI?**

(This question targets specific technical differences between the two systems.)

2. **How do LangChain's agents differ from those used by CrewAI, and what are their respective use cases?**

(This question focuses on the agent architecture and its applications in each system.)

3. **What benefits or drawbacks does LangChain have when it comes to memory allocation compared to CrewAI?**

(This question looks at the practical implications of these technical differences on performance or usability.)

4. **Are there any notable advantages or disadvantages of using agents with LangChain versus CrewAI, and how do these impact overall system design?**

(This question explores the broader implications of agent usage on system architecture and overall functionalit

In [10]:
# Step 4: QA chain per sub-question
qa_prompt = PromptTemplate.from_template("""
Use the context below to answer the question.

Context:
{context}

Question: {input}
""")
qa_chain = create_stuff_documents_chain(llm=llm, prompt=qa_prompt)

RAG Pipeline Logic


In [14]:
def full_query_decomposition_rag_pipeline(user_query):
    sub_qs_text=decomposition_chain.invoke({"question":user_query})
    sub_questions=[q.strip("-•1234567890. ").strip() for q in sub_qs_text.split('\n') if q.strip()]
    print("sub_questions",sub_questions)
    results=[]
    for subq in sub_questions:
        docs=retriever.invoke(subq)
        result=qa_chain.invoke({"input":subq,"context":docs})
        results.append(f"Q: {subq}\nA {result}")
    return "\n\n".join(results)

In [15]:
query="How does LangChain use memory and agents compared to CrewAI?"
final_answer=full_query_decomposition_rag_pipeline(query)
print("Final Answer")
print(final_answer)

sub_questions ['Here are 3 smaller sub-questions that can help with retrieving relevant information about LangChain, its approach to memory usage and agent utilization, and how it compares to CrewAI:', '"What is the primary method of memory management used by LangChain?"', '"How does LangChain utilize agents in its architecture, and what benefits or drawbacks does this approach offer compared to other methods?"', '"In comparison to CrewAI, how does LangChain\'s memory usage and agent utilization patterns differ, and what are the implications of these differences for tasks like natural language processing and knowledge graph querying?"', "These sub-questions can help narrow down the search and retrieve more relevant information about LangChain's approach to memory and agents, as well as its comparison with CrewAI"]
Final Answer
Q: Here are 3 smaller sub-questions that can help with retrieving relevant information about LangChain, its approach to memory usage and agent utilization, and h